# Fundamentos de Machine Learning con `scikit-learn` — Sesión 1  
## Bloque 1: fundamentos, flujo de trabajo y API mental de sklearn  
## Bloque 2: regresión supervisada, evaluación y selección de modelos

**Objetivo del cuaderno**  

La idea no es solo escribir código que "funcione", sino entender:

- qué problema estamos resolviendo,
- cómo se organiza el trabajo en `scikit-learn`,
- qué diferencia hay entre **transformadores** y **predictores**,
- por qué usamos `train_test_split`, validación cruzada y `GridSearchCV`,
- y cómo interpretar métricas y resultados en un problema real de **regresión**.

Vamos a trabajar con un dataset público y asequible sobre coches y consumo: el conocido conjunto de datos **Auto MPG**.  
Nuestro objetivo será predecir la variable **`mpg`** (*miles per gallon*), que mide la eficiencia del coche.

---

## Cómo leer este notebook

Este es un cuaderno **deliberadamente narrado**.  
No está pensado para leerlo "a toda velocidad", sino para ir parando en las explicaciones y preguntarnos constantemente:

- ¿qué estamos haciendo?
- ¿por qué lo hacemos así?
- ¿qué aprende el modelo?
- ¿qué podría salir mal?
- ¿qué nos dice cada métrica?

En otras palabras: queremos aprender **Machine Learning** y también aprender a trabajar con **`scikit-learn`** con cabeza.

## Hoja de ruta

### Parte A — Bloque 1
1. Qué es Machine Learning y qué tipos de problemas existen.
2. Qué significa *aprender a partir de datos*.
3. Flujo de trabajo típico en ML.
4. API mental de `scikit-learn`.
5. Transformadores vs predictores.
6. Módulos típicos de `scikit-learn`.
7. `train_test_split`, validación cruzada y búsqueda de hiperparámetros.

### Parte B — Bloque 2
8. Qué es una regresión.
9. Dataset Auto MPG.
10. Análisis inicial y preparación de variables.
11. Baseline muy simple.
12. Regresión lineal.
13. Métricas de regresión.
14. Validación cruzada.
15. Modelo con vecinos (`KNN`) y escalado.
16. `GridSearchCV`.
17. Ejemplo de *feature engineering* con `PolynomialFeatures`.
18. Comparación final de modelos.

# Parte A. Fundamentos de ML y `scikit-learn`

## 1. ¿Qué es Machine Learning?

Una forma muy razonable de explicarlo es esta:

> En Machine Learning no programamos una lista cerrada de reglas del tipo  
> "si ocurre A, entonces haz B".  
> En lugar de eso, proporcionamos **datos** y un **objetivo**, y el algoritmo ajusta internamente unos parámetros para aprender una relación entre entradas y salidas.

Por ejemplo:

- Si quiero predecir el precio de una casa a partir de metros cuadrados, barrio, año de construcción, etc., estoy ante un problema de **regresión**.
- Si quiero decidir si un correo es spam o no, estoy ante un problema de **clasificación**.
- Si quiero agrupar clientes por perfiles de comportamiento sin etiquetas previas, estoy ante un problema de **clustering**.

En este notebook nos centraremos en **aprendizaje supervisado** y, concretamente, en **regresión**.

### Aprendizaje supervisado
En aprendizaje supervisado tenemos:

- unas **variables de entrada** (también llamadas *features* o predictores),
- y una **variable objetivo** (*target*) que queremos predecir.

En notación típica:

- `X`: matriz de características
- `y`: variable objetivo

En nuestro caso:
- `X` contendrá información del coche,
- `y` será `mpg`.

## 2. Tipos de problemas de ML, muy resumidos

### Regresión
La salida es una **cantidad numérica continua**.

Ejemplos:
- consumo de un coche,
- precio de una vivienda,
- demanda energética,
- temperatura mañana a las 12:00.

### Clasificación
La salida es una **clase** o etiqueta.

Ejemplos:
- spam / no spam,
- fraude / no fraude,
- gato / perro / pájaro.

### Clustering
No tenemos etiquetas; intentamos encontrar grupos.

### Reducción de dimensionalidad
Intentamos representar los datos con menos variables preservando la información importante.

---

Hoy trabajamos sobre todo con las dos ideas de la primera mitad del curso:

- **flujo general de trabajo en ML**,  
- **regresión supervisada**.

## 3. El flujo de trabajo típico en Machine Learning

Aunque luego cada proyecto tenga matices, una forma sana de pensar un proyecto de ML es la siguiente:

1. **Entender el problema**
   - ¿Qué quiero predecir?
   - ¿Qué significa hacerlo bien?
   - ¿Qué error me importa más?

2. **Conseguir y entender los datos**
   - ¿Qué columnas tengo?
   - ¿Qué significan?
   - ¿Hay valores ausentes?
   - ¿Hay columnas poco útiles o problemáticas?

3. **Separar `X` e `y`**
   - `X`: lo que le damos al modelo.
   - `y`: lo que queremos predecir.

4. **Separar entrenamiento y prueba**
   - El conjunto de entrenamiento sirve para ajustar el modelo.
   - El conjunto de prueba sirve para una evaluación final honesta.

5. **Preprocesar**
   - imputar nulos,
   - escalar variables,
   - codificar categóricas,
   - crear nuevas variables si tiene sentido.

6. **Entrenar uno o varios modelos**

7. **Evaluar**
   - con métricas adecuadas,
   - sin autoengañarnos,
   - comparando contra un baseline.

8. **Ajustar hiperparámetros y validar**
   - por ejemplo con validación cruzada y `GridSearchCV`.

9. **Interpretar resultados**
   - ¿el modelo mejora de verdad?
   - ¿es estable?
   - ¿tiene sentido práctico?

### Dos ideas importantísimas
- Un modelo puede ir muy bien en entrenamiento y mal en datos nuevos: eso es **sobreajuste** (*overfitting*).
- Un único `train/test split` es útil, pero la **validación cruzada** nos da una visión más robusta.

## 4. La API mental de `scikit-learn`

Una de las grandes virtudes de `scikit-learn` es que muchos objetos siguen una interfaz coherente.

### Métodos que merece la pena memorizar pronto

#### `fit`
Ajusta el objeto con datos de entrenamiento.

- En un modelo de predicción, `fit` aprende parámetros.
- En un transformador, `fit` aprende cómo debe transformar los datos.

#### `transform`
Transforma los datos.

Ejemplos:
- estandarizar variables,
- imputar nulos,
- generar variables polinómicas,
- codificar variables categóricas.

#### `fit_transform`
Hace `fit` y luego `transform`.  
Es muy cómodo en entrenamiento, pero hay que usarlo con cuidado para no mezclar información de train y test.

#### `predict`
Genera predicciones.

#### `score`
Devuelve una métrica por defecto del objeto.  
**Ojo**: para aprender y para evaluar con criterio, casi siempre preferiremos usar funciones de `sklearn.metrics` o `cross_validate`, no depender solo de `score`.

---

### Idea central
En `scikit-learn`, gran parte del trabajo consiste en combinar objetos que siguen esta lógica común.

## 5. Transformadores vs predictores

Esta distinción es clave.

### Transformador
Un transformador toma unos datos de entrada y los convierte en otra representación.

Ejemplos:
- `SimpleImputer`
- `StandardScaler`
- `OneHotEncoder`
- `PolynomialFeatures`

Los transformadores suelen tener:
- `fit`
- `transform`
- a veces `fit_transform`

### Predictor
Un predictor intenta aprender una relación entre `X` e `y` para luego hacer predicciones.

Ejemplos:
- `LinearRegression`
- `KNeighborsRegressor`
- `RandomForestRegressor`

Los predictores suelen tener:
- `fit`
- `predict`

### ¿Por qué es tan importante entender esto?
Porque en un flujo real rara vez entrenamos el modelo sobre los datos "tal cual".  
Normalmente necesitamos:

1. transformar datos,
2. y después predecir.

Por eso objetos como `Pipeline` o `ColumnTransformer` son tan importantes: nos permiten **encadenar** transformadores y predictores de forma ordenada y segura.

## 6. Módulos típicos de `scikit-learn`

No hace falta memorizar todos, pero sí tener un mapa mental.

- `sklearn.datasets`  
  Cargar datasets de ejemplo o utilidades relacionadas con datos.

- `sklearn.model_selection`  
  `train_test_split`, validación cruzada, `GridSearchCV`, etc.

- `sklearn.preprocessing`  
  Escalado, codificación, transformaciones.

- `sklearn.impute`  
  Imputación de valores ausentes.

- `sklearn.compose`  
  Especialmente útil `ColumnTransformer` para aplicar transformaciones distintas según columnas.

- `sklearn.pipeline`  
  Para encadenar pasos: transformadores + predictor.

- `sklearn.metrics`  
  Métricas y funciones de evaluación.

- `sklearn.dummy`  
  Modelos "tontos" pero muy útiles como baseline.

- `sklearn.linear_model`, `sklearn.neighbors`, `sklearn.tree`, `sklearn.ensemble`, ...  
  Familias de modelos.

---

La buena noticia es que, aunque haya muchos módulos, la filosofía de uso es bastante homogénea.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.ensemble import RandomForestRegressor

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

print("Entorno listo.")

## 7. Un microejemplo para interiorizar la API

Antes de lanzarnos al dataset real, hagamos un ejemplo diminuto.  
La idea no es resolver un problema importante, sino **mirar con lupa la mecánica de sklearn**.

In [ ]:
X_toy = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])
y_toy = np.array([2.0, 4.1, 6.2, 8.1, 10.2])

scaler = StandardScaler()
X_toy_scaled = scaler.fit_transform(X_toy)

model = LinearRegression()
model.fit(X_toy_scaled, y_toy)

pred_toy = model.predict(X_toy_scaled)

print("X original:")
print(X_toy.ravel())
print("\nX transformada con StandardScaler:")
print(np.round(X_toy_scaled.ravel(), 3))
print("\nPredicciones del modelo:")
print(np.round(pred_toy, 3))

### ¿Qué acaba de pasar?

1. Hemos creado un **transformador**: `StandardScaler()`.
2. Le hemos hecho `fit_transform(X_toy)`.
   - `fit`: aprende media y desviación típica de la columna.
   - `transform`: aplica la estandarización.
3. Hemos creado un **predictor**: `LinearRegression()`.
4. Le hemos hecho `fit(X_toy_scaled, y_toy)`.
   - ahora aprende una relación entre la entrada y la salida.
5. Con `predict(...)` obtenemos predicciones.

Este patrón va a repetirse constantemente.

---

## Una advertencia importante
Cuando trabajamos con conjuntos de entrenamiento y prueba, **nunca** deberíamos ajustar el escalador usando también datos de prueba.  
Es decir:

- correcto: `fit` del escalador en `X_train`, luego `transform` en `X_train` y `X_test`.
- incorrecto: `fit_transform` sobre todo el dataset antes de separar train/test.

¿Por qué? Porque eso introduce **fuga de información** (*data leakage*).

## 8. `Pipeline`: la forma profesional de unir pasos

Hacer el escalado "a mano" ayuda a entender qué ocurre.  
Pero en un proyecto real suele ser mejor encapsular el flujo con `Pipeline`.

Ventajas:

- el código queda más limpio,
- se reducen errores,
- ayuda a evitar *data leakage*,
- se integra muy bien con validación cruzada y `GridSearchCV`.

Primero veamos un ejemplo pequeño.

In [ ]:
toy_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

toy_pipeline.fit(X_toy, y_toy)
pred_toy_pipe = toy_pipeline.predict(X_toy)

print("Predicciones con Pipeline:")
print(np.round(pred_toy_pipe, 3))

La idea es potente:

- los pasos intermedios deben ser **transformadores**,
- el último paso suele ser un **predictor**.

Esto encaja perfectamente con la distinción transformadores vs predictores.

## 9. `train_test_split`, validación cruzada y `GridSearchCV`

Antes de ir al caso real, conviene fijar tres conceptos.

### `train_test_split`
Separa datos en:
- **train**: para entrenar,
- **test**: para evaluar al final.

Sirve para simular el escenario real: entrenamos con unos datos y luego vemos qué pasa en datos no vistos.

### Validación cruzada
En lugar de depender de una única partición, repetimos el proceso varias veces en distintos pliegues (*folds*).  
Nos da una evaluación más robusta.

### `GridSearchCV`
Busca automáticamente la mejor combinación de hiperparámetros dentro de una rejilla de candidatos usando validación cruzada.

---

### Parámetros vs hiperparámetros
- **Parámetros**: los aprende el modelo durante `fit`.
  - por ejemplo, los coeficientes de una regresión lineal.
- **Hiperparámetros**: los elegimos nosotros antes del entrenamiento.
  - por ejemplo, `n_neighbors` en KNN.

# Parte B. Regresión supervisada con un dataset real

## 10. El problema que vamos a resolver

Queremos predecir **`mpg`** (*miles per gallon*).  
Es decir: queremos estimar la eficiencia en consumo de un coche usando características como:

- cilindrada,
- potencia,
- peso,
- aceleración,
- año del modelo,
- procedencia.

### ¿Por qué esto es una regresión?
Porque la salida que buscamos es numérica y continua.  
No queremos clasificar el coche en categorías; queremos estimar un valor.

## 11. Cargar el dataset público Auto MPG

Usaremos una copia local `mpg.csv` si está disponible en la misma carpeta que el notebook.  
Si no, el código intentará leerlo desde una URL pública.

> Esto hace el notebook más cómodo para clase: si te llevas el CSV junto al `.ipynb`, todo debería funcionar igual.

In [ ]:
DATA_CANDIDATES = [
    Path("mpg.csv"),
    Path("/mnt/data/mpg.csv")
]
DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"

data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)

if data_path is not None:
    df = pd.read_csv(data_path)
    print(f"Dataset cargado desde: {data_path}")
else:
    df = pd.read_csv(DATA_URL)
    print("Dataset cargado desde URL pública")

print(f"Forma del dataset: {df.shape}")
df.head()

## 12. Primer vistazo a los datos

Nunca entrenamos un modelo "a ciegas".  
Antes conviene mirar:

- tipos de datos,
- valores ausentes,
- rango y significado general de las columnas,
- si hay columnas que no queramos usar.

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False)

### Observaciones iniciales

- `mpg` será nuestro **target**.
- `horsepower` tiene algunos valores ausentes.
- `origin` es categórica.
- `name` contiene texto libre con el nombre del coche.

### Decisión didáctica
Para esta primera sesión:

- **sí** usaremos `origin`, porque nos permite ver cómo tratar variables categóricas;
- **no** usaremos `name` como predictor, porque abre la puerta a procesamiento de texto, y hoy no es el objetivo.

En otras palabras: queremos aprender ML tabular y el ecosistema básico de sklearn, no NLP.

In [ ]:
df.describe(include="all").T

## 13. Un poco de análisis exploratorio

No necesitamos hacer un EDA gigantesco, pero sí una exploración razonable.

Buscamos:
- cómo se distribuye `mpg`,
- qué variables parecen relacionadas con ella,
- si hay relaciones potencialmente no lineales,
- si existen valores atípicos llamativos.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["mpg"].dropna(), bins=20, edgecolor="black")
ax.set_title("Distribución de mpg")
ax.set_xlabel("mpg")
ax.set_ylabel("Frecuencia")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.scatterplot(data=df, x="weight", y="mpg", ax=axes[0])
axes[0].set_title("mpg vs weight")

sns.scatterplot(data=df, x="horsepower", y="mpg", ax=axes[1])
axes[1].set_title("mpg vs horsepower")

sns.scatterplot(data=df, x="displacement", y="mpg", ax=axes[2])
axes[2].set_title("mpg vs displacement")

plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ["mpg", "cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year"]
corr = df[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlaciones numéricas")
plt.show()

### Lectura rápida de estas gráficas

Se intuyen varias cosas:

- `weight`, `horsepower` y `displacement` parecen relacionarse de forma inversa con `mpg`;
- `model_year` suele tener relación positiva con `mpg`;
- algunas relaciones no parecen perfectamente lineales;
- las escalas de las variables son muy distintas.

Todo esto nos prepara para decisiones posteriores:
- modelos lineales,
- modelos basados en distancia como KNN,
- y modelos más flexibles como bosques aleatorios.

## 14. Definir `X` e `y`

Aquí aparece una de las convenciones más importantes de `scikit-learn`:

- `X`: estructura tabular con predictores,
- `y`: objetivo.

Vamos a quitar `name` para centrarnos en variables tabulares clásicas.

In [ ]:
target = "mpg"
features = [col for col in df.columns if col not in [target, "name"]]

X = df[features].copy()
y = df[target].copy()

print("Columnas de X:")
print(features)
print("\nDimensión de X:", X.shape)
print("Dimensión de y:", y.shape)

## 15. Separar entrenamiento y prueba con `train_test_split`

Vamos a reservar un conjunto de prueba final.

### ¿Por qué ahora?
Porque a partir de aquí queremos ser disciplinados:

- todo lo que "aprendamos" del dato para preprocesar debe ajustarse usando solo el train;
- el test debe quedar lo más virgen posible hasta el final.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

## 16. Baseline: comparemos contra algo muy simple

Una regla muy sana en ML es no enamorarse del primer modelo.  
Antes, conviene tener un **baseline**.

`DummyRegressor` no intenta entender ninguna relación real.  
Por ejemplo, con la estrategia `"mean"` siempre predice la media de `y_train`.

Puede parecer ridículo, pero pedagógicamente es oro:
- si un modelo sofisticado no mejora claramente al baseline,
- entonces quizá el modelo no está aportando gran cosa.

In [ ]:
dummy_reg = DummyRegressor(strategy="mean")
dummy_reg.fit(X_train, y_train)

dummy_pred = dummy_reg.predict(X_test)

dummy_mae = mean_absolute_error(y_test, dummy_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_pred))
dummy_r2 = r2_score(y_test, dummy_pred)

print("Baseline DummyRegressor")
print(f"MAE : {dummy_mae:.3f}")
print(f"RMSE: {dummy_rmse:.3f}")
print(f"R²  : {dummy_r2:.3f}")

### Recordatorio de métricas de regresión

#### MAE — Mean Absolute Error
Promedio del error absoluto.

#### MSE — Mean Squared Error
Promedio del error al cuadrado.

#### RMSE — Root Mean Squared Error
Raíz del MSE.  
Está en las mismas unidades que el target y penaliza más los errores grandes.

#### R²
Mide cuánta variabilidad del target explica el modelo.

- `1.0` sería perfecto.
- `0.0` equivale aproximadamente a predecir la media.
- puede incluso ser negativo si el modelo lo hace muy mal.

## 17. Separar columnas numéricas y categóricas

Aquí empezamos a usar de verdad una de las grandes ideas de sklearn:
aplicar transformaciones distintas según el tipo de columna.

In [ ]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)

## 18. Transformadores de preprocesado

Vamos a construir dos pequeños flujos:

### Pipeline numérico
1. `SimpleImputer(strategy="median")`
2. `StandardScaler()`

### Pipeline categórico
1. `SimpleImputer(strategy="most_frequent")`
2. `OneHotEncoder(handle_unknown="ignore")`

Y luego los uniremos con `ColumnTransformer`.

In [ ]:
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_preprocessor, numeric_features),
    ("cat", categorical_preprocessor, categorical_features)
])

preprocessor

### Aquí vuelve a aparecer la idea de transformador

Fíjate que:
- `SimpleImputer`, `StandardScaler` y `OneHotEncoder` son transformadores;
- `ColumnTransformer` también actúa como transformador global;
- y más tarde pondremos un predictor al final del `Pipeline`.

Esta arquitectura resume muy bien el estilo de `scikit-learn`.

## 19. Primer modelo serio: Regresión Lineal

La regresión lineal suele ser un punto de partida excelente porque es:
- simple,
- rápida,
- interpretable,
- y suele dar un baseline fuerte en muchos problemas tabulares.

In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_pred)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_pred))
linear_r2 = r2_score(y_test, linear_pred)

print("Regresión lineal")
print(f"MAE : {linear_mae:.3f}")
print(f"RMSE: {linear_rmse:.3f}")
print(f"R²  : {linear_r2:.3f}")

## 20. Comparación visual: valores reales vs predichos

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=linear_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle="--")
plt.xlabel("Valores reales")
plt.ylabel("Predicciones")
plt.title("Regresión lineal: reales vs predichos")
plt.show()

## 21. Un detalle importante sobre `Pipeline`

Cuando hacemos:

```python
linear_model.fit(X_train, y_train)
```

internamente ocurre algo así:

1. el `preprocessor` hace `fit` sobre `X_train`,
2. transforma `X_train`,
3. el regresor se entrena con la versión transformada.

Y cuando hacemos:

```python
linear_model.predict(X_test)
```

el pipeline:
1. transforma `X_test` usando lo aprendido en train,
2. y luego predice.

## 22. Validación cruzada: una evaluación más robusta

Hasta ahora hemos usado una única partición train/test.  
Eso está bien para empezar, pero una sola partición puede ser "afortunada" o "desafortunada".

Por eso existe la **validación cruzada**.

In [ ]:
cv_rmse_scores = cross_val_score(
    linear_model,
    X_train,
    y_train,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

cv_rmse_scores = -cv_rmse_scores

print("RMSE en cada fold:", np.round(cv_rmse_scores, 3))
print("RMSE medio CV     :", round(cv_rmse_scores.mean(), 3))
print("Desviación típica :", round(cv_rmse_scores.std(), 3))

### ¿Por qué aparece en negativo?
Porque sklearn organiza muchas rutinas para **maximizar** scores.  
Como RMSE es un error que queremos minimizar, se devuelve con signo negativo.

## 23. `cross_validate`: varias métricas a la vez

In [ ]:
scoring = {
    "mae": "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2": "r2"
}

cv_results_linear = cross_validate(
    linear_model,
    X_train,
    y_train,
    cv=5,
    scoring=scoring,
    return_train_score=False
)

cv_summary_linear = pd.DataFrame({
    "MAE": -cv_results_linear["test_mae"],
    "RMSE": -cv_results_linear["test_rmse"],
    "R2": cv_results_linear["test_r2"]
})

cv_summary_linear

In [ ]:
cv_summary_linear.agg(["mean", "std"]).T

## 24. Segundo modelo: KNN Regressor

KNN predice mirando los vecinos más cercanos de un punto en el espacio de características.

Es un gran ejemplo de modelo para recordar por qué el **escalado** importa.

In [ ]:
knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", KNeighborsRegressor())
])

knn_pipeline.fit(X_train, y_train)
knn_pred = knn_pipeline.predict(X_test)

knn_mae = mean_absolute_error(y_test, knn_pred)
knn_rmse = np.sqrt(mean_squared_error(y_test, knn_pred))
knn_r2 = r2_score(y_test, knn_pred)

print("KNN baseline")
print(f"MAE : {knn_mae:.3f}")
print(f"RMSE: {knn_rmse:.3f}")
print(f"R²  : {knn_r2:.3f}")

## 25. `GridSearchCV`: buscar hiperparámetros con criterio

¿Cómo elegimos `n_neighbors` en KNN?  
No deberíamos decidirlo "a ojo" mirando el test.

La respuesta sana es:
- usar solo el train,
- probar varias combinaciones,
- usar validación cruzada.

In [ ]:
param_grid_knn = {
    "regressor__n_neighbors": list(range(2, 21)),
    "regressor__weights": ["uniform", "distance"],
    "regressor__p": [1, 2]
}

grid_knn = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid_knn,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid_knn.fit(X_train, y_train)

print("Mejores hiperparámetros KNN:")
print(grid_knn.best_params_)
print(f"Mejor RMSE CV: {-grid_knn.best_score_:.3f}")

In [ ]:
best_knn = grid_knn.best_estimator_
best_knn_pred = best_knn.predict(X_test)

best_knn_mae = mean_absolute_error(y_test, best_knn_pred)
best_knn_rmse = np.sqrt(mean_squared_error(y_test, best_knn_pred))
best_knn_r2 = r2_score(y_test, best_knn_pred)

print("KNN optimizado con GridSearchCV")
print(f"MAE : {best_knn_mae:.3f}")
print(f"RMSE: {best_knn_rmse:.3f}")
print(f"R²  : {best_knn_r2:.3f}")

### Idea metodológica importante

Fíjate en el esquema correcto:

1. separo **train** y **test**,
2. hago `GridSearchCV` **solo sobre train**,
3. elijo el mejor modelo según CV,
4. evalúo una única vez en **test**.

## 26. Ejemplo de *feature engineering*: `PolynomialFeatures`

La relación entre `horsepower` y `mpg` no parece perfectamente lineal.  
Vamos a construir un mini problema con una sola variable para ilustrar:

- entrada: `horsepower`
- salida: `mpg`

Y compararemos regresión lineal simple frente a variables polinómicas.

In [ ]:
X_hp = df[["horsepower"]].copy()
y_hp = df["mpg"].copy()

X_hp_train, X_hp_test, y_hp_train, y_hp_test = train_test_split(
    X_hp, y_hp,
    test_size=0.2,
    random_state=RANDOM_STATE
)

poly_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

param_grid_poly = {
    "poly__degree": [1, 2, 3, 4, 5]
}

grid_poly = GridSearchCV(
    estimator=poly_pipeline,
    param_grid=param_grid_poly,
    scoring="neg_root_mean_squared_error",
    cv=5
)

grid_poly.fit(X_hp_train, y_hp_train)

print("Mejor grado polinómico:", grid_poly.best_params_["poly__degree"])
print("Mejor RMSE CV:", round(-grid_poly.best_score_, 3))

In [ ]:
best_poly = grid_poly.best_estimator_
poly_pred = best_poly.predict(X_hp_test)

poly_mae = mean_absolute_error(y_hp_test, poly_pred)
poly_rmse = np.sqrt(mean_squared_error(y_hp_test, poly_pred))
poly_r2 = r2_score(y_hp_test, poly_pred)

print("Regresión polinómica (solo con horsepower)")
print(f"MAE : {poly_mae:.3f}")
print(f"RMSE: {poly_rmse:.3f}")
print(f"R²  : {poly_r2:.3f}")

In [ ]:
x_line = np.linspace(X_hp["horsepower"].min(), X_hp["horsepower"].max(), 300)
x_line_df = pd.DataFrame({"horsepower": x_line})

y_line = best_poly.predict(x_line_df)

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="horsepower", y="mpg", alpha=0.6)
plt.plot(x_line, y_line, linewidth=3, label="Modelo polinómico")
plt.title("Relación entre horsepower y mpg")
plt.legend()
plt.show()

### ¿Qué nos enseña este experimento?

- `PolynomialFeatures` actúa como **transformador**;
- `LinearRegression` actúa como **predictor**;
- mejorar la representación de las variables puede ser tan importante como cambiar de algoritmo.

## 27. Tercer modelo: Random Forest Regressor

Los bosques aleatorios son un modelo de conjunto (*ensemble*) basado en muchos árboles de decisión.

Ventajas:
- capturan no linealidades e interacciones,
- suelen ir bien en tabular,
- no requieren escalado para funcionar razonablemente bien.

In [ ]:
rf_model = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_features)
    ])),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest")
print(f"MAE : {rf_mae:.3f}")
print(f"RMSE: {rf_rmse:.3f}")
print(f"R²  : {rf_r2:.3f}")

## 28. Tabla comparativa final en test

In [ ]:
results_test = pd.DataFrame([
    {"Modelo": "DummyRegressor (mean)", "MAE": dummy_mae, "RMSE": dummy_rmse, "R2": dummy_r2},
    {"Modelo": "LinearRegression + preprocessing", "MAE": linear_mae, "RMSE": linear_rmse, "R2": linear_r2},
    {"Modelo": "KNN baseline", "MAE": knn_mae, "RMSE": knn_rmse, "R2": knn_r2},
    {"Modelo": "KNN optimizado con GridSearchCV", "MAE": best_knn_mae, "RMSE": best_knn_rmse, "R2": best_knn_r2},
    {"Modelo": "RandomForestRegressor", "MAE": rf_mae, "RMSE": rf_rmse, "R2": rf_r2},
])

results_test.sort_values("RMSE")

In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(data=results_test.sort_values("RMSE"), x="RMSE", y="Modelo")
plt.title("Comparación de modelos por RMSE (menor es mejor)")
plt.show()

## 29. ¿Qué hemos aprendido realmente en esta sesión?

### Sobre Machine Learning
- Un modelo no es magia: aprende patrones a partir de datos.
- Hay que separar entrenamiento y evaluación.
- Una métrica aislada no cuenta toda la historia.
- Un baseline simple es imprescindible.

### Sobre `scikit-learn`
- La API tiene una lógica coherente:
  - `fit`,
  - `transform`,
  - `predict`.
- Los **transformadores** cambian la representación del dato.
- Los **predictores** generan predicciones.
- `Pipeline` y `ColumnTransformer` son piezas clave.
- `train_test_split`, `cross_validate` y `GridSearchCV` forman parte del flujo estándar.

## 30. Trampas comunes que ya deberías saber detectar

1. Escalar antes del split.
2. Elegir hiperparámetros mirando el test.
3. Quedarse solo con una métrica.
4. Pensar que un modelo más complejo siempre es mejor.
5. Confundir parámetros con hiperparámetros.

## 31. Resumen ejecutivo para llevarse a casa

1. **ML es un proceso, no solo un algoritmo.**
2. En `scikit-learn`, entender **transformar** vs **predecir** te ahorra mucha confusión.
3. El patrón profesional habitual es:
   - separar train/test,
   - construir un pipeline,
   - validar,
   - ajustar hiperparámetros,
   - evaluar al final.
4. En regresión, no basta con entrenar: hay que mirar métricas y compararlas con un baseline.

## 32. Ejercicios propuestos para clase o para casa

### Ejercicio 1
Quita la columna `origin` y vuelve a entrenar la regresión lineal.  
¿Mejora o empeora?

### Ejercicio 2
Prueba a cambiar la estrategia del `DummyRegressor` a `"median"`.

### Ejercicio 3
Amplía la rejilla de `GridSearchCV` para KNN.

### Ejercicio 4
Haz un `GridSearchCV` pequeño para `RandomForestRegressor`.

### Ejercicio 5
Repite el experimento polinómico usando `weight` en lugar de `horsepower`.

## 33. Cierre

Con esto cerramos una primera sesión bastante completa:

- hemos introducido la lógica general de ML,
- hemos aprendido a pensar en términos de `X` e `y`,
- hemos visto cómo se organiza `scikit-learn`,
- y hemos entrenado y evaluado varios modelos de regresión de forma razonablemente seria.

En la siguiente parte natural del curso, una vez dominada esta base, podremos entrar en clasificación con mucha más tranquilidad.